# Bulk Takeover Paginated Reports + Workspace App Setup

**Problems this notebook solves:**
1. Paginated reports deployed by the SP aren't connecting to data sources (SP owns them but SSO/credentials aren't flowing)
2. Need to take ownership in bulk instead of one-by-one in the UI
3. Deployed reports need to be included in the workspace Power BI App

**How it works:**
- When run by a **user** in a Fabric notebook, the TakeOver call transfers ownership to YOUR identity (the logged-in user), which enables SSO pass-through to the SQL endpoint
- When run by a **service principal**, it keeps SP ownership but re-configures credentials via the Gateway Datasource API
- The App section creates or updates the workspace app to include all paginated reports

**Run this in the target workspace** (UAT or Prod) after a deployment.

## 1. Configuration

In [ ]:
# === CONFIGURATION ===
# Set the target workspace ID (from the deployment log or Fabric portal URL)
WORKSPACE_ID = "68c48a5c-8b88-4559-962d-5dc7f29ab96a"  # UAT workspace

# Filter: set to None to process ALL paginated reports, or a list of names to target specific ones
TARGET_REPORTS = None  # e.g., ["Anna Asset Delivery Dates", "Aged Debtors"]

# Set to True to also update gateway datasource credentials after takeover
UPDATE_CREDENTIALS = True

# App configuration
UPDATE_APP = True  # Set to True to create/update the workspace app
APP_AUDIENCE = []  # List of security group object IDs or email addresses for app access
# e.g., APP_AUDIENCE = ["finance-team@company.com", "00000000-0000-0000-0000-000000000000"]

# API base URLs
FABRIC_API = "https://api.fabric.microsoft.com/v1"
PBI_API = "https://api.powerbi.com/v1.0/myorg"

print(f"Target workspace: {WORKSPACE_ID}")
print(f"Target reports: {'ALL' if TARGET_REPORTS is None else TARGET_REPORTS}")
print(f"Update credentials: {UPDATE_CREDENTIALS}")
print(f"Update app: {UPDATE_APP}")

## 2. Authentication

In [ ]:
import requests
import json
import time

# In a Fabric notebook, use the built-in credential
try:
    from notebookutils import mssparkutils
    access_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
    print("\u2713 Authenticated via Fabric notebook token")
except ImportError:
    # Fallback: if running locally, use azure-identity
    from azure.identity import DefaultAzureCredential
    credential = DefaultAzureCredential()
    token = credential.get_token("https://analysis.windows.net/powerbi/api/.default")
    access_token = token.token
    print("\u2713 Authenticated via DefaultAzureCredential")

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

def api_get(url):
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.ok:
        return resp.json()
    print(f"  \u2717 GET {resp.status_code}: {resp.text[:300]}")
    return None

def api_post(url, body=None):
    resp = requests.post(url, headers=headers, json=body, timeout=60)
    return resp

def api_patch(url, body=None):
    resp = requests.patch(url, headers=headers, json=body, timeout=60)
    return resp

# Verify access
ws_info = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}")
if ws_info:
    print(f"\u2713 Connected to workspace: {ws_info.get('name', 'unknown')}")

# Show who we are
me = api_get(f"{PBI_API}/profile")
if me:
    print(f"\u2713 Running as: {me.get('displayName', me.get('emailAddress', 'unknown'))}")
else:
    print("\u2139\ufe0f Running as service principal (no user profile)")

## 3. List All Paginated Reports + Current Ownership

In [ ]:
print("=" * 80)
print("LISTING ALL PAGINATED REPORTS")
print("=" * 80)

# Get reports from both APIs for a complete picture
pbi_reports = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/reports")
paginated_reports = []

if pbi_reports:
    for r in pbi_reports.get("value", []):
        if r.get("reportType") == "PaginatedReport":
            paginated_reports.append(r)

print(f"\nFound {len(paginated_reports)} paginated report(s)\n")
print(f"{'#':>3}  {'Report Name':<55} {'Dataset ID':<38} {'Owner'}")
print("-" * 150)

for i, r in enumerate(sorted(paginated_reports, key=lambda x: x.get("name", "")), 1):
    name = r.get("name", "N/A")
    ds_id = r.get("datasetId", "N/A")
    created_by = r.get("createdBy", "unknown")
    modified_by = r.get("modifiedBy", "unknown")
    
    # Check if this report is in our target list
    targeted = "" if TARGET_REPORTS is None else (" <<<" if name in TARGET_REPORTS else "")
    print(f"{i:3d}  {name:<55} {ds_id:<38} {modified_by}{targeted}")

# Store for use in next cells
all_paginated = paginated_reports

## 4. Check Data Source Status (Before Takeover)

In [ ]:
print("=" * 80)
print("DATA SOURCE STATUS (PRE-TAKEOVER)")
print("=" * 80)

reports_to_process = all_paginated
if TARGET_REPORTS is not None:
    reports_to_process = [r for r in all_paginated if r.get("name") in TARGET_REPORTS]
    print(f"\nFiltered to {len(reports_to_process)} targeted report(s)\n")

ds_status = []  # Store status for each report

for r in reports_to_process:
    name = r.get("name", "N/A")
    report_id = r["id"]
    
    print(f"\n\u2500 {name} (id={report_id})")
    
    # Get datasources via PBI API
    ds_info = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/reports/{report_id}/datasources")
    if ds_info:
        sources = ds_info.get("value", [])
        for ds in sources:
            gw_id = ds.get("gatewayId", "none")
            ds_id = ds.get("datasourceId", "none")
            ds_type = ds.get("datasourceType", "unknown")
            conn = ds.get("connectionDetails", {})
            server = conn.get("server", "N/A")
            database = conn.get("database", "N/A")
            print(f"  \u2514 type={ds_type}, server={server}, db={database}")
            print(f"    gatewayId={gw_id}, datasourceId={ds_id}")
            ds_status.append({
                "report_name": name,
                "report_id": report_id,
                "gateway_id": gw_id,
                "datasource_id": ds_id,
                "server": server,
                "database": database,
                "ds_type": ds_type
            })
    else:
        print(f"  \u26a0\ufe0f Could not retrieve datasources (report may need takeover first)")
        ds_status.append({
            "report_name": name,
            "report_id": report_id,
            "gateway_id": None,
            "datasource_id": None,
            "error": True
        })

print(f"\n\n\u2705 Scanned {len(reports_to_process)} report(s), found {len(ds_status)} data source(s)")

## 5. Bulk Takeover - Take Ownership of All Paginated Reports
This calls `POST /groups/{id}/reports/{id}/Default.TakeOver` for each paginated report.

**When run by a user in a Fabric notebook:** ownership transfers to YOU (the logged-in user), enabling SSO pass-through to the SQL endpoint.

**When run by an SP:** the SP retains ownership and credentials are reconfigured.

In [ ]:
print("=" * 80)
print("BULK TAKEOVER: PAGINATED REPORTS")
print("=" * 80)

takeover_results = {"success": 0, "already_owned": 0, "failed": 0, "failed_reports": []}

for r in reports_to_process:
    name = r.get("name", "N/A")
    report_id = r["id"]
    
    print(f"  Taking over: {name}...", end=" ")
    
    resp = api_post(f"{PBI_API}/groups/{WORKSPACE_ID}/reports/{report_id}/Default.TakeOver")
    
    if resp.status_code == 200:
        print("\u2713 Done")
        takeover_results["success"] += 1
    elif resp.status_code == 400 and "AlreadyOwner" in resp.text:
        print("\u2713 Already owned")
        takeover_results["already_owned"] += 1
    else:
        print(f"\u2717 Failed ({resp.status_code}: {resp.text[:100]})")
        takeover_results["failed"] += 1
        takeover_results["failed_reports"].append(name)

print("\n" + "=" * 80)
print(f"RESULTS: {takeover_results['success']} succeeded, "
      f"{takeover_results['already_owned']} already owned, "
      f"{takeover_results['failed']} failed")
print("=" * 80)

if takeover_results["failed_reports"]:
    print(f"\n\u26a0\ufe0f Failed reports:")
    for name in takeover_results["failed_reports"]:
        print(f"   - {name}")

## 6. (Optional) Update Gateway Data Source Credentials
After takeover, this sets OAuth2 credentials with `useCallerAADIdentity=true` on each report's gateway datasource so SSO pass-through works.

**Skip this cell if takeover alone fixed the data source connectivity.**

In [ ]:
if not UPDATE_CREDENTIALS:
    print("Skipping credential update (UPDATE_CREDENTIALS = False)")
else:
    print("=" * 80)
    print("UPDATING GATEWAY DATA SOURCE CREDENTIALS")
    print("=" * 80)
    
    cred_results = {"success": 0, "skipped": 0, "failed": 0}
    
    for r in reports_to_process:
        name = r.get("name", "N/A")
        report_id = r["id"]
        
        # Re-fetch datasources after takeover
        ds_info = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/reports/{report_id}/datasources")
        if not ds_info:
            print(f"  \u2717 {name}: Could not get datasources")
            cred_results["failed"] += 1
            continue
        
        sources = ds_info.get("value", [])
        for ds in sources:
            gw_id = ds.get("gatewayId")
            ds_id = ds.get("datasourceId")
            
            if not gw_id or not ds_id:
                print(f"  \u2139\ufe0f {name}: No gateway/datasource ID — skipping")
                cred_results["skipped"] += 1
                continue
            
            # Update credentials: OAuth2 with caller AAD identity pass-through
            cred_payload = {
                "credentialDetails": {
                    "credentialType": "OAuth2",
                    "credentials": "{\"credentialData\":\"\"}",
                    "encryptedConnection": "Encrypted",
                    "encryptionAlgorithm": "None",
                    "privacyLevel": "Organizational",
                    "useCallerAADIdentity": True,
                    "useEndUserOAuth2Credentials": False
                }
            }
            
            url = f"{PBI_API}/gateways/{gw_id}/datasources/{ds_id}"
            resp = api_patch(url, cred_payload)
            
            if resp.ok:
                print(f"  \u2713 {name}: Credentials updated (OAuth2 + CallerAADIdentity)")
                cred_results["success"] += 1
            else:
                print(f"  \u2717 {name}: Credential update failed ({resp.status_code}: {resp.text[:150]})")
                cred_results["failed"] += 1
    
    print(f"\n\u2705 Credentials: {cred_results['success']} updated, "
          f"{cred_results['skipped']} skipped, {cred_results['failed']} failed")

## 7. Verify Data Source Status (After Takeover)

In [ ]:
print("=" * 80)
print("DATA SOURCE STATUS (POST-TAKEOVER)")
print("=" * 80)

for r in reports_to_process:
    name = r.get("name", "N/A")
    report_id = r["id"]
    
    print(f"\n\u2500 {name}")
    
    ds_info = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/reports/{report_id}/datasources")
    if ds_info:
        for ds in ds_info.get("value", []):
            conn = ds.get("connectionDetails", {})
            gw_id = ds.get("gatewayId", "none")
            print(f"  \u2514 type={ds.get('datasourceType')}, server={conn.get('server', 'N/A')}, "
                  f"db={conn.get('database', 'N/A')}, gatewayId={gw_id}")
    else:
        print(f"  \u26a0\ufe0f Still cannot retrieve datasources")

# Also check Fabric connections
print("\n" + "-" * 80)
print("FABRIC ITEM CONNECTIONS (post-takeover):")
print("-" * 80)
for r in reports_to_process[:5]:  # Check first 5 as a sample
    name = r.get("name", "N/A")
    report_id = r["id"]
    conns = api_get(f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{report_id}/connections")
    if conns:
        for c in conns.get("value", []):
            print(f"  {name}: connectivityType={c.get('connectivityType')}, "
                  f"id={c.get('id', 'N/A')}")
    else:
        print(f"  {name}: no connections visible")

---
# Part 2: Workspace App Management

## 8. Check Existing Workspace App
Power BI workspace apps are published from the workspace. This checks if an app already exists.

In [ ]:
print("=" * 80)
print("WORKSPACE APP STATUS")
print("=" * 80)

# List all apps the current user/SP has access to
apps = api_get(f"{PBI_API}/apps")

workspace_app = None
if apps:
    for app in apps.get("value", []):
        if app.get("workspaceId") == WORKSPACE_ID:
            workspace_app = app
            break

if workspace_app:
    print(f"\n\u2713 App exists for this workspace:")
    print(f"  Name: {workspace_app.get('name', 'N/A')}")
    print(f"  App ID: {workspace_app.get('id', 'N/A')}")
    print(f"  Last Update: {workspace_app.get('lastUpdate', 'N/A')}")
    
    # List app reports
    app_id = workspace_app["id"]
    app_reports = api_get(f"{PBI_API}/apps/{app_id}/reports")
    if app_reports:
        print(f"\n  Reports in app: {len(app_reports.get('value', []))}")
        pg_in_app = [r for r in app_reports.get("value", []) if r.get("reportType") == "PaginatedReport"]
        print(f"  Paginated reports in app: {len(pg_in_app)}")
        
        # Show which paginated reports are NOT in the app
        app_report_names = {r.get("name") for r in app_reports.get("value", [])}
        workspace_pg_names = {r.get("name") for r in all_paginated}
        missing = workspace_pg_names - app_report_names
        
        if missing:
            print(f"\n  \u26a0\ufe0f Paginated reports NOT in app ({len(missing)}):")
            for name in sorted(missing):
                print(f"    - {name}")
        else:
            print(f"  \u2713 All paginated reports are in the app")
else:
    print("\n\u26a0\ufe0f No app found for this workspace.")
    print("   You can create one in Cell 9.")

## 9. Create or Update the Workspace App

This uses the Power BI REST API to publish/update the workspace app, including all paginated reports.

**API Notes:**
- `POST /groups/{id}/CreateApp` creates a new app from the workspace
- `POST /groups/{id}/UpdateApp` updates an existing app
- Both accept an `UpdateAppRequest` body where you can configure which content is included and the audience
- The `reports` section controls which reports are visible in the app
- All workspace content is included by default unless you explicitly exclude items

In [ ]:
if not UPDATE_APP:
    print("Skipping app update (UPDATE_APP = False)")
else:
    print("=" * 80)
    print("CREATE / UPDATE WORKSPACE APP")
    print("=" * 80)
    
    # Get ALL reports in workspace (both regular and paginated)
    all_reports = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/reports")
    all_datasets = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets")
    all_dashboards = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/dashboards")
    
    report_list = all_reports.get("value", []) if all_reports else []
    dataset_list = all_datasets.get("value", []) if all_datasets else []
    dashboard_list = all_dashboards.get("value", []) if all_dashboards else []
    
    print(f"\nWorkspace content to include in app:")
    print(f"  Reports: {len(report_list)} (including paginated)")
    print(f"  Datasets: {len(dataset_list)}")
    print(f"  Dashboards: {len(dashboard_list)}")
    
    # Build the update body - include all content
    # Reports section: mark all as included and visible in navigation
    report_configs = []
    for r in report_list:
        report_configs.append({
            "id": r["id"],
            "isIncluded": True,
            "isVisibleInNavigation": True
        })
    
    # Build audience config
    access_list = []
    for entry in APP_AUDIENCE:
        if "@" in entry:
            # Email address = individual user
            access_list.append({
                "identifier": entry,
                "principalType": "User",
                "appUserAccessRight": "ReadExplore"
            })
        else:
            # GUID = security group
            access_list.append({
                "identifier": entry,
                "principalType": "Group",
                "appUserAccessRight": "ReadExplore"
            })
    
    app_body = {
        "updateAppSettings": {
            "publishDetails": {
                "reports": report_configs
            }
        }
    }
    
    # Only add access list if non-empty
    if access_list:
        app_body["updateAppSettings"]["access"] = access_list
    
    # Decide: Create or Update
    if workspace_app:
        print(f"\nUpdating existing app '{workspace_app.get('name')}'...")
        resp = api_post(f"{PBI_API}/groups/{WORKSPACE_ID}/UpdateApp", app_body)
    else:
        print("\nCreating new app from workspace...")
        resp = api_post(f"{PBI_API}/groups/{WORKSPACE_ID}/CreateApp", app_body)
    
    if resp.ok or resp.status_code == 200:
        print(f"\u2713 App {'updated' if workspace_app else 'created'} successfully!")
        print(f"  All {len(report_configs)} report(s) (including paginated) are now in the app.")
        if not access_list:
            print(f"\n  \u2139\ufe0f NOTE: No audience configured (APP_AUDIENCE is empty).")
            print(f"       Set APP_AUDIENCE in Cell 1 and re-run to grant access.")
            print(f"       Or configure audience manually in the Fabric portal.")
    else:
        print(f"\u2717 App operation failed: {resp.status_code}")
        try:
            print(json.dumps(resp.json(), indent=2))
        except Exception:
            print(resp.text[:500])

## 10. Verify App Contents (Post-Update)

In [ ]:
print("=" * 80)
print("APP VERIFICATION")
print("=" * 80)

# Re-fetch apps to pick up newly created app
apps = api_get(f"{PBI_API}/apps")
workspace_app = None
if apps:
    for app in apps.get("value", []):
        if app.get("workspaceId") == WORKSPACE_ID:
            workspace_app = app
            break

if not workspace_app:
    print("\u26a0\ufe0f No app found. It may take a moment to appear after creation.")
else:
    app_id = workspace_app["id"]
    print(f"\nApp: {workspace_app.get('name')} (id={app_id})")
    
    app_reports = api_get(f"{PBI_API}/apps/{app_id}/reports")
    if app_reports:
        report_list = app_reports.get("value", [])
        regular = [r for r in report_list if r.get("reportType") != "PaginatedReport"]
        paginated = [r for r in report_list if r.get("reportType") == "PaginatedReport"]
        
        print(f"\nTotal reports in app: {len(report_list)}")
        print(f"  Regular reports: {len(regular)}")
        print(f"  Paginated reports: {len(paginated)}")
        
        if paginated:
            print(f"\nPaginated reports in app:")
            for i, r in enumerate(sorted(paginated, key=lambda x: x.get("name", "")), 1):
                print(f"  {i:3d}. {r.get('name', 'N/A')}")
        
        # Cross-check against workspace
        app_pg_names = {r.get("name") for r in paginated}
        ws_pg_names = {r.get("name") for r in all_paginated}
        missing = ws_pg_names - app_pg_names
        
        if missing:
            print(f"\n\u26a0\ufe0f Still missing from app ({len(missing)}):")
            for name in sorted(missing):
                print(f"   - {name}")
        else:
            print(f"\n\u2705 All {len(ws_pg_names)} workspace paginated reports are in the app!")

---
# Part 3: Pipeline Integration Guidance

## 11. Generate Pipeline Post-Deployment Script

This cell generates the Python code you can add to your deployment pipeline to automate:
1. Takeover of paginated reports after deployment
2. App update to include new reports

**Important considerations:**
- **Takeover by SP**: If the pipeline SP takes ownership, the SP becomes the data source owner. This works IF the SP has access to the data source (e.g., via a shareable cloud connection or gateway credentials configured for the SP).
- **Takeover by User**: If you need a *user* to own the reports (for SSO), the takeover must be run interactively (e.g., this notebook), not from the pipeline SP.
- **App update**: Can be automated from the pipeline SP if the SP has admin access to the workspace.

In [ ]:
pipeline_code = '''
# =========================================================================
# Post-deployment steps to add to deploy_artifacts.py
# Add these methods to the FabricDeployer class
# =========================================================================

def _post_deploy_takeover_paginated_reports(self):
    """
    After all paginated reports are deployed, take ownership of each one
    so the SP can manage data source credentials.
    
    This is already partially done per-report in _deploy_paginated_report().
    This method is a catch-all for any that were missed.
    """
    logger.info("Post-deploy: Verifying paginated report ownership...")
    
    pbi_url = f"https://api.powerbi.com/v1.0/myorg/groups/{self.workspace_id}/reports"
    headers = self.client.auth.get_auth_headers()
    
    resp = requests.get(pbi_url, headers=headers, timeout=60)
    if not resp.ok:
        logger.warning(f"Could not list reports for takeover: {resp.status_code}")
        return
    
    reports = resp.json().get("value", [])
    paginated = [r for r in reports if r.get("reportType") == "PaginatedReport"]
    
    for r in paginated:
        self.client.take_over_paginated_report(self.workspace_id, r["id"])


def _post_deploy_update_app(self):
    """
    After deployment, update the workspace app to include all reports.
    Creates the app if it doesn't exist.
    """
    logger.info("Post-deploy: Updating workspace app...")
    
    headers = self.client.auth.get_auth_headers()
    pbi_base = "https://api.powerbi.com/v1.0/myorg"
    
    # Get all reports
    resp = requests.get(
        f"{pbi_base}/groups/{self.workspace_id}/reports",
        headers=headers, timeout=60
    )
    if not resp.ok:
        logger.warning(f"Could not list reports for app update: {resp.status_code}")
        return
    
    reports = resp.json().get("value", [])
    report_configs = [
        {"id": r["id"], "isIncluded": True, "isVisibleInNavigation": True}
        for r in reports
    ]
    
    body = {
        "updateAppSettings": {
            "publishDetails": {
                "reports": report_configs
            }
        }
    }
    
    # Try Update first (existing app), fall back to Create
    resp = requests.post(
        f"{pbi_base}/groups/{self.workspace_id}/UpdateApp",
        headers=headers, json=body, timeout=60
    )
    if resp.ok:
        logger.info("App updated successfully with all reports")
    elif resp.status_code == 404:
        resp = requests.post(
            f"{pbi_base}/groups/{self.workspace_id}/CreateApp",
            headers=headers, json=body, timeout=60
        )
        if resp.ok:
            logger.info("App created successfully with all reports")
        else:
            logger.warning(f"App creation failed: {resp.status_code}: {resp.text[:200]}")
    else:
        logger.warning(f"App update failed: {resp.status_code}: {resp.text[:200]}")
'''

print(pipeline_code)
print("\n" + "=" * 80)
print("USAGE:")
print("  Call these methods at the end of deploy_all() in deploy_artifacts.py")
print("  e.g.:")
print("    self._post_deploy_takeover_paginated_reports()")
print("    self._post_deploy_update_app()")
print("=" * 80)
print("\nNOTE: The SP takeover keeps the SP as data source owner.")
print("If reports need USER ownership for SSO, run this notebook manually")
print("after each deployment instead of automating in the pipeline.")

## Summary

| Task | How | Automatable in Pipeline? |
|------|-----|------------------------|
| **Bulk takeover** | Cell 5 — calls `Default.TakeOver` on each report | Yes, but ownership goes to SP. If user SSO needed, run this notebook manually. |
| **Update credentials** | Cell 6 — sets OAuth2 + CallerAADIdentity on gateway datasources | Yes, via pipeline SP |
| **Include in app** | Cells 9-10 — calls `UpdateApp`/`CreateApp` with all reports | Yes, via pipeline SP (needs workspace admin) |
| **App audience** | Cell 1 `APP_AUDIENCE` — set groups/users | Yes, pass in body of `UpdateApp` |

**Key insight:** The paginated report data source issue is because the **SP takes ownership** during deployment, but the SP's identity may not have SSO/pass-through configured for the Fabric SQL endpoint. Solutions:
1. **Best:** Configure a ShareableCloud connection in Fabric and set `paginated_report_connection` in the config — the deployment already handles this
2. **Quick fix:** Run this notebook as a user after deployment to transfer ownership
3. **Automated:** Add the pipeline code from Cell 11 + ensure SP has gateway credential access